<a href="https://colab.research.google.com/github/bamiboy237/embedding_benchmark/blob/main/notebooks/MTEB_Retrieval_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
requirements = [
    "sentence-transformers",
    "mteb",
    "langchain",
    "langchain-text-splitters",
    "langchain-community",
    "faiss-cpu",
    "pandas",
    "numpy",
    "scikit-learn",
    "matplotlib",
    "plotly",
    "tqdm",
    "pydantic",
    "mteb"
]

for req in requirements:
    !pip install {req}

In [ ]:
# importing necessary libraries
import pandas as pd
from pathlib import Path
from typing import Optional
import json

input_data = Path("/workspaces/embedding_benchmark/data/processed/chunks.jsonl")
df = pd.read_json(input_data, lines=True)

df.head()

In [ ]:
# load and run evaluation
import mteb

models = [
    "nvidia/llama-nemotron-embed-1b-v2",
    "Qwen/Qwen3-Embedding-4B",
    "Linq-AI-Research/Linq-Embed-Mistral",
    "intfloat/multilingual-e5-large-instruct",
    "intfloat/e5-mistral-7b-instruct",
    "Qwen/Qwen3-Embedding-0.6B",
    "sentence-transformers/all-MiniLM-L6-v2",
    "intfloat/e5-small-v2",
    "intfloat/e5-base-instruct",
    "intfloat/e5-large-instruct",

]

model_name = ""

if "7b" in model_name.lower() or "mistral" in model_name.lower():
    batch_size = 2  
elif "4b" in model_name.lower():
    batch_size = 8  
else:
    batch_size = 64 
    
encode_kwargs = {"batch_size": batch_size, "normalize_embeddings": True}

tasks = mteb.get_tasks(task_types=["Retrieval"], domains=["Legal"])
model = mteb.get_model(model_name)
cache="/workspaces/embedding_benchmark/cache"

results = mteb.evaluate(model, tasks=tasks, encode_kwargs=encode_kwargs, cache=cache, prediction_folder="/workspaces/embedding_benchmark/predictions")



In [ ]:
# retrieve results
from mteb.cache import ResultCache

cache = ResultCache(cache)
results = cache.load_results(models, tasks)

df = results.to_dataframe()